In [2]:
%pip install pandas numpy scikit-learn matplotlib pyarrow -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import os, json, csv, time, random, logging
from datetime import date, datetime, timedelta
import sqlite3

random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
print("✓ Environment ready for: Python Requests and Pagination")

✓ Environment ready for: Python Requests and Pagination


In [5]:
# Cell 2: Generate Dataset
# Simulate paginated API response data
api_records = []
for page in range(1, 6):  # 5 pages
    for i in range(100):  # 100 per page
        idx = (page-1)*100 + i
        api_records.append({
            'id': idx+1,
            'name': f'Supplier_{idx+1}',
            'country': random.choice(['US','UK','DE','JP','IN','BR']),
            'category': random.choice(['Electronics','Textiles','Chemicals','Food','Metals']),
            'annual_revenue': round(random.uniform(100000, 50000000), 2),
            'rating': round(random.uniform(1, 5), 1),
            'is_active': random.choice([True, True, True, False])
        })
df = pd.DataFrame(api_records)

print(f"✓ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✓ Dataset loaded: 500 rows, 7 columns


,id,name,country,category,annual_revenue,rating,is_active
0,1,Supplier_1,BR,Food,8558137.09,2.1,True
1,2,Supplier_2,JP,Textiles,3257030.80,2.7,False
2,3,Supplier_3,DE,Food,39155064.55,2.4,True
3,4,Supplier_4,BR,Food,29730939.26,1.8,True
4,5,Supplier_5,UK,Chemicals,437691.73,2.6,False


In [6]:
# Cell 3: Data Profiling
print('=' * 60)
print('DATA PROFILE: Python Requests and Pagination')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: Python Requests and Pagination

Shape: (500, 7)

Column Types:
id                  int64
name                  str
country               str
category              str
annual_revenue    float64
rating            float64
is_active            bool
dtype: object

Null Counts:
id                0
name              0
country           0
category          0
annual_revenue    0
rating            0
is_active         0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
               id  annual_revenue      rating
count  500.000000    5.000000e+02  500.000000
mean   250.500000    2.593185e+07    2.984200
std    144.481833    1.418406e+07    1.142217
min      1.000000    1.046081e+05    1.000000
25%    125.750000    1.369350e+07    2.000000
50%    250.500000    2.697797e+07    2.950000
75%    375.250000    3.827676e+07    4.000000
max    500.000000    4.989838e+07    5.000000

Unique Values:
  id: 500 unique
  name: 500 unique
  country: 6 unique
  category: 5 unique
  annual_revenue

In [7]:
# Cell 4: Core Implementation — Python Requests and Pagination

import time
import logging
import random
import json

random.seed(42)

# ============================================================
# Pagination and Retry Logic — simulated without external APIs
# ============================================================

class MockPaginatedAPI:
    """Simulates a paginated API with rate limiting and transient errors."""

    def __init__(self, total_records=500, per_page=100):
        self.data = [
            {'id': i, 'name': f'Item_{i}', 'value': round(random.uniform(10, 1000), 2)}
            for i in range(1, total_records + 1)
        ]
        self.per_page = per_page
        self.total_pages = (total_records + per_page - 1) // per_page
        self.call_count = 0

    def get(self, page=1):
        """Simulate GET request with occasional errors."""
        self.call_count += 1
        # Simulate rate limiting (HTTP 429) on every 7th call
        if self.call_count % 7 == 0:
            return {'status_code': 429, 'headers': {'Retry-After': '1'}, 'body': None}
        # Simulate server error (HTTP 503) on every 13th call
        if self.call_count % 13 == 0:
            return {'status_code': 503, 'body': None}
        # Normal response
        start = (page - 1) * self.per_page
        end = start + self.per_page
        records = self.data[start:end]
        return {
            'status_code': 200,
            'body': {
                'data': records,
                'pagination': {
                    'page': page,
                    'per_page': self.per_page,
                    'total': len(self.data),
                    'total_pages': self.total_pages
                }
            }
        }


def fetch_all_with_retry(api, max_retries=3, backoff_base=0.1):
    """
    Fetch all pages from a paginated API with retry logic.

    Handles:
    - HTTP 429 (rate limit): wait and retry
    - HTTP 503 (server error): exponential backoff
    - Connection errors: retry up to max_retries
    """
    all_records = []
    page = 1
    total_pages = None

    while True:
        retries = 0
        success = False

        while retries <= max_retries and not success:
            try:
                resp = api.get(page=page)

                if resp['status_code'] == 429:
                    # Rate limited — wait as instructed
                    wait = float(resp.get('headers', {}).get('Retry-After', 1))
                    logging.warning(f'Page {page}: Rate limited (429). Waiting {wait}s...')
                    time.sleep(wait * 0.01)  # Scaled down for demo
                    retries += 1
                    continue

                if resp['status_code'] == 503:
                    # Server error — exponential backoff
                    wait = backoff_base * (2 ** retries)
                    logging.warning(f'Page {page}: Server error (503). Backoff {wait:.2f}s (attempt {retries+1})')
                    time.sleep(wait * 0.01)  # Scaled down for demo
                    retries += 1
                    continue

                if resp['status_code'] == 200:
                    body = resp['body']
                    records = body['data']
                    pagination = body['pagination']
                    total_pages = pagination['total_pages']

                    if not records:
                        logging.info(f'Page {page}: Empty response, stopping.')
                        return all_records

                    all_records.extend(records)
                    logging.info(f'Page {page}/{total_pages}: Fetched {len(records)} records '
                                 f'(total so far: {len(all_records)})')
                    success = True

            except Exception as e:
                retries += 1
                logging.error(f'Page {page}: Error — {e} (attempt {retries})')
                if retries > max_retries:
                    logging.error(f'Page {page}: Max retries exceeded. Stopping.')
                    return all_records

        if not success:
            logging.error(f'Page {page}: Failed after {max_retries} retries. Stopping.')
            break

        # Check if we have fetched all pages
        if total_pages and page >= total_pages:
            break
        page += 1

    return all_records


# --- Execute the paginated fetch ---
print('=== Paginated API Fetch with Retry Logic ===')
mock_api = MockPaginatedAPI(total_records=500, per_page=100)
results = fetch_all_with_retry(mock_api, max_retries=3)

print(f'\nTotal records fetched: {len(results)}')
print(f'API calls made: {mock_api.call_count}')
print(f'Sample records:')
df_fetched = pd.DataFrame(results)
print(df_fetched.head(10))

# Summary statistics
print(f'\nValue statistics:')
print(f'  Min: {df_fetched["value"].min():.2f}')
print(f'  Max: {df_fetched["value"].max():.2f}')
print(f'  Mean: {df_fetched["value"].mean():.2f}')

print('\n-- Python Requests and Pagination implementation complete')

2026-04-01 13:47:30,948 [INFO] Page 1/5: Fetched 100 records (total so far: 100)
2026-04-01 13:47:30,949 [INFO] Page 2/5: Fetched 100 records (total so far: 200)
2026-04-01 13:47:30,950 [INFO] Page 3/5: Fetched 100 records (total so far: 300)
2026-04-01 13:47:30,950 [INFO] Page 4/5: Fetched 100 records (total so far: 400)
2026-04-01 13:47:30,950 [INFO] Page 5/5: Fetched 100 records (total so far: 500)


=== Paginated API Fetch with Retry Logic ===

Total records fetched: 500
API calls made: 5
Sample records:
   id     name   value
0   1   Item_1  643.03
1   2   Item_2   34.76
2   3   Item_3  282.28
3   4   Item_4  230.98
4   5   Item_5  739.11
5   6   Item_6  679.93
6   7   Item_7  893.26
7   8   Item_8   96.07
8   9   Item_9  427.70
9  10  Item_10   39.50

Value statistics:
  Min: 10.40
  Max: 999.29
  Mean: 506.43

-- Python Requests and Pagination implementation complete


In [8]:
# Cell 5: Results Analysis
print('=' * 60)
print('RESULTS: Python Requests and Pagination')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nTop 5 rows of results:')
print(df.head())

RESULTS: Python Requests and Pagination
Input rows: 500
Processing complete

Top 5 rows of results:
   id        name country   category  annual_revenue  rating  is_active
0   1  Supplier_1      BR       Food      8558137.09     2.1       True
1   2  Supplier_2      JP   Textiles      3257030.80     2.7      False
2   3  Supplier_3      DE       Food     39155064.55     2.4       True
3   4  Supplier_4      BR       Food     29730939.26     1.8       True
4   5  Supplier_5      UK  Chemicals       437691.73     2.6      False


In [9]:
# Cell 6: Self-Check — Pagination
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — Pagination')
print('=' * 50)

checks = {
    'DataFrame exists and is not empty': len(df) > 0,
    'Has at least 2 columns': len(df.columns) >= 2,
    'No fully-null columns': df.isnull().mean().max() < 0.5,
    'Has at least 10 rows': len(df) >= 10,
    'Data types look valid': df.dtypes is not None,
}

passed = sum(1 for v in checks.values() if v)
for name, ok in checks.items():
    print(f'  {"PASS" if ok else "FAIL"}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')

SELF-CHECK — Pagination
  PASS: DataFrame exists and is not empty
  PASS: Has at least 2 columns
  PASS: No fully-null columns
  PASS: Has at least 10 rows
  PASS: Data types look valid

5/5 self-checks passed

All good! Click "Run Tests" to submit for official validation.


In [10]:
# ============================================
# Try-It-Out: Paginated API Harvester
# ============================================

import pandas as pd
import time
import random
import logging

logging.basicConfig(level=logging.INFO)

# --------------------------------------------
# Mock Offset-Based API
# --------------------------------------------

class MockOffsetAPI:
    def __init__(self, total=8400, limit=100):
        self.total = total
        self.limit = limit
        self.call_count = 0

        self.data = [
            {
                "product_id": i,
                "name": f"Product_{i}",
                "price": round(random.uniform(10, 5000), 2),
                "category": random.choice(["Electronics", "Clothing", "Home", "Sports"]),
                "in_stock": random.choice([True, False])
            }
            for i in range(total)
        ]

    def get(self, offset=0, limit=100):
        self.call_count += 1

        # Simulate rate limit (429)
        if self.call_count % 10 == 0:
            return {
                "status_code": 429,
                "headers": {"Retry-After": "1"},
                "body": None
            }

        end = offset + limit
        return {
            "status_code": 200,
            "body": {
                "data": self.data[offset:end],
                "total": self.total,
                "offset": offset,
                "limit": limit
            }
        }

# --------------------------------------------
# Harvester Function
# --------------------------------------------

def harvest_all_products(api, limit=100, max_retries=5):
    offset = 0
    all_data = []
    total = None

    while True:
        retries = 0

        while retries <= max_retries:
            resp = api.get(offset=offset, limit=limit)

            if resp["status_code"] == 429:
                wait = float(resp["headers"].get("Retry-After", 1))
                logging.warning(f"Rate limited. Waiting {wait}s...")
                time.sleep(wait * 0.01)
                retries += 1
                continue

            if resp["status_code"] == 200:
                body = resp["body"]
                data = body["data"]

                if total is None:
                    total = body["total"]

                all_data.extend(data)

                logging.info(f"Fetched {len(all_data)}/{total}")

                break

        if retries > max_retries:
            logging.error("Max retries exceeded")
            break

        offset += limit

        if offset >= total:
            break

    return all_data

# --------------------------------------------
# Execute Harvester
# --------------------------------------------

print("=== Running API Harvester ===")

api = MockOffsetAPI()
results = harvest_all_products(api)

df_products = pd.DataFrame(results)

print(f"\nTotal records fetched: {len(df_products)}")
print(f"API calls made: {api.call_count}")

# --------------------------------------------
# Save to Parquet
# --------------------------------------------

output_path = "products.parquet"
df_products.to_parquet(output_path, index=False)

print(f"Saved to {output_path}")

df_products.head()

2026-04-01 13:49:19,974 [INFO] Fetched 100/8400
2026-04-01 13:49:19,975 [INFO] Fetched 200/8400
2026-04-01 13:49:19,977 [INFO] Fetched 300/8400
2026-04-01 13:49:19,979 [INFO] Fetched 400/8400
2026-04-01 13:49:19,982 [INFO] Fetched 500/8400
2026-04-01 13:49:19,984 [INFO] Fetched 600/8400
2026-04-01 13:49:19,985 [INFO] Fetched 700/8400
2026-04-01 13:49:19,985 [INFO] Fetched 800/8400
2026-04-01 13:49:19,986 [INFO] Fetched 900/8400
2026-04-01 13:49:19,986 [WARNING] Rate limited. Waiting 1.0s...
2026-04-01 13:49:19,997 [INFO] Fetched 1000/8400
2026-04-01 13:49:19,998 [INFO] Fetched 1100/8400
2026-04-01 13:49:20,003 [INFO] Fetched 1200/8400
2026-04-01 13:49:20,004 [INFO] Fetched 1300/8400
2026-04-01 13:49:20,007 [INFO] Fetched 1400/8400
2026-04-01 13:49:20,008 [INFO] Fetched 1500/8400
2026-04-01 13:49:20,008 [INFO] Fetched 1600/8400
2026-04-01 13:49:20,009 [INFO] Fetched 1700/8400
2026-04-01 13:49:20,009 [INFO] Fetched 1800/8400
2026-04-01 13:49:20,009 [WARNING] Rate limited. Waiting 1.0s...

=== Running API Harvester ===


2026-04-01 13:49:20,138 [INFO] Fetched 6400/8400
2026-04-01 13:49:20,138 [INFO] Fetched 6500/8400
2026-04-01 13:49:20,139 [INFO] Fetched 6600/8400
2026-04-01 13:49:20,139 [INFO] Fetched 6700/8400
2026-04-01 13:49:20,139 [INFO] Fetched 6800/8400
2026-04-01 13:49:20,141 [INFO] Fetched 6900/8400
2026-04-01 13:49:20,141 [INFO] Fetched 7000/8400
2026-04-01 13:49:20,141 [INFO] Fetched 7100/8400
2026-04-01 13:49:20,141 [INFO] Fetched 7200/8400
2026-04-01 13:49:20,142 [WARNING] Rate limited. Waiting 1.0s...
2026-04-01 13:49:20,153 [INFO] Fetched 7300/8400
2026-04-01 13:49:20,153 [INFO] Fetched 7400/8400
2026-04-01 13:49:20,154 [INFO] Fetched 7500/8400
2026-04-01 13:49:20,154 [INFO] Fetched 7600/8400
2026-04-01 13:49:20,157 [INFO] Fetched 7700/8400
2026-04-01 13:49:20,157 [INFO] Fetched 7800/8400
2026-04-01 13:49:20,158 [INFO] Fetched 7900/8400
2026-04-01 13:49:20,160 [INFO] Fetched 8000/8400
2026-04-01 13:49:20,160 [INFO] Fetched 8100/8400
2026-04-01 13:49:20,162 [WARNING] Rate limited. Waitin


Total records fetched: 8400
API calls made: 93
Saved to products.parquet


,product_id,name,price,category,in_stock
0,0,Product_0,3115.06,Electronics,False
1,1,Product_1,2963.54,Electronics,True
2,2,Product_2,3217.32,Clothing,False
3,3,Product_3,916.83,Home,False
4,4,Product_4,1643.49,Sports,False


In [11]:
# --------------------------------------------
# Extended Validation Checks
# --------------------------------------------

print("=" * 50)
print("EXTENDED VALIDATION CHECKS")
print("=" * 50)

checks = {
    "All records fetched (8400)": len(df_products) == 8400,
    "No duplicate product_id": df_products["product_id"].nunique() == len(df_products),
    "Parquet file created": os.path.exists("products.parquet"),
}

passed = sum(checks.values())

for name, result in checks.items():
    print(f"{'PASS' if result else 'FAIL'}: {name}")

print(f"\n{passed}/{len(checks)} checks passed")

EXTENDED VALIDATION CHECKS
PASS: All records fetched (8400)
PASS: No duplicate product_id
PASS: Parquet file created

3/3 checks passed
